In [62]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage,HumanMessage,ToolMessage
from langchain_core.prompts import SystemMessagePromptTemplate,HumanMessagePromptTemplate,ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [63]:
load_dotenv(override=True)
parser = StrOutputParser()

In [64]:
llm = ChatOpenAI(model="gpt-5.4-mini")
llm_groq = ChatGroq(temperature=0,model="openai/gpt-oss-120b",api_key=os.getenv("GROQ_API_KEY"))

In [65]:
# result = llm.invoke("please tell smallest poem in english")
# print(result.content)

In [66]:
# system_message = SystemMessage("You are a  english teacher")
# user_message = HumanMessage("Tell me about english only one sentence")
# messages = [system_message,user_message]
# result = llm.invoke(messages)
# print(result.content)

In [67]:
# system_prompt = SystemMessagePromptTemplate.from_template("You write emails as {sender_role} to {receiver_role}. Tone: {tone}. Max {words} words.")
# user_prompt = HumanMessagePromptTemplate.from_template("Write an email about {topic}")

# messages = [system_prompt,user_prompt]
# template = ChatPromptTemplate(messages)

# quection = template.invoke({"sender_role":"project manager","receiver_role":"client","tone":"polite, apologetic","words":100,"topic":"project delivery delayed by one week"})
# # print(quection)
# result = llm.invoke(quection)
# print(result.content)

In [68]:
system_prompt = SystemMessagePromptTemplate.from_template("You write emails as {sender_role} to {receiver_role}. Tone: {tone}. Max {words} words.")
user_prompt = HumanMessagePromptTemplate.from_template("Write an email about {topic}")

messages = [system_prompt,user_prompt]
template = ChatPromptTemplate(messages)
chain = template | llm | parser

# result = chain.invoke({"sender_role":"project manager","receiver_role":"client","tone":"polite, apologetic","words":100,"topic":"project delivery delayed by one week"})
# print(result)

In [69]:
from operator import itemgetter
translate_prompt = ChatPromptTemplate.from_template("Translate {email} this language {target}")
# translate_chain = translate_prompt | llm | parser

compose_chain = (
    {"email":chain,"target": itemgetter("target")} | translate_prompt |  llm | parser
)
output = compose_chain.invoke({"sender_role":"project manager","receiver_role":"client","tone":"polite, apologetic","words":100,"topic":"project delivery delayed by one week","target":"telugu"})

# output = translate_chain.invoke({"email":result,"target":"telugu"})
print(output)


విషయం: ప్రాజెక్ట్ డెలివరీ టైమ్‌లైన్‌పై అప్డేట్

ప్రియమైన [Client Name],

ప్రాజెక్ట్ డెలివరీ ఒక వారం ఆలస్యం అవుతుందని తెలియజేయడం పట్ల నాకు చింతగా ఉంది. అనుకోని కొన్ని సమస్యలు ఎదురుకావడంతో వాటిని పరిష్కరించడానికి అదనపు సమయం అవసరమైంది. తుది డెలివరబుల్ మా నాణ్యత ప్రమాణాలను చేరేలా ఉండాలని మేము కోరుకుంటున్నాము.

ఈ అసౌకర్యానికి మేము హృదయపూర్వకంగా క్షమాపణలు కోరుతున్నాము మరియు మీ సహనం, అవగాహనకు ధన్యవాదాలు తెలియజేస్తున్నాము. మేము పనిని వీలైనంత త్వరగా పూర్తి చేయడానికి చురుకుగా ప్రయత్నిస్తున్నాము మరియు పురోగతిపై మీకు నిరంతరం సమాచారం అందిస్తాము.

శుభాకాంక్షలతో,  
[Your Name]  
ప్రాజెక్ట్ మేనేజర్


In [70]:
# system_prompt = SystemMessagePromptTemplate.from_template("You are a chef. The user is {diet}. Never use {avoid}")
# user_prompt = HumanMessagePromptTemplate.from_template("Give me a recipe for {dish}")

# messages = [system_prompt,user_prompt]

# template = ChatPromptTemplate(messages)
# chain = template | llm | parser

In [71]:
from operator import itemgetter

system_prompt = SystemMessagePromptTemplate.from_template("You are a chef. The user is {diet}. Never use {avoid}")
user_prompt = HumanMessagePromptTemplate.from_template("Give me a recipe for {dish}")

messages = [system_prompt,user_prompt]

template = ChatPromptTemplate(messages)
chain = template | llm | parser

extract_ingredients_prompt =  ChatPromptTemplate.from_template("List only the ingredients with quantities from this recipe, one per line:{recipe}")
shoping_list_prompt =  ChatPromptTemplate.from_template("Group these ingredients by store section (Produce, Dairy, Bakery, Pantry, Other).Use markdown headings and bullets:{ingredients}")

extract_chain = (
    {"recipe":chain} |
    extract_ingredients_prompt |
    llm |
    parser 
    

)

composite_chain = (
        {"ingredients":extract_chain} |
    shoping_list_prompt |
    llm |
    parser 
)

output = composite_chain.invoke({
    "diet": "vegan",
    "avoid": "nuts",
    "dish": "Banoffee Pie",
})

print(output)


## Produce
- 3–4 ripe bananas, sliced
- 2–3 tbsp lemon juice

## Dairy
- 1 can full-fat coconut milk, chilled overnight
- 1 cup vegan whipped cream

## Bakery
- 250 g vegan digestive biscuits or graham crackers

## Pantry
- 100 g vegan butter, melted
- 1 tbsp brown sugar, optional
- 1/2 cup brown sugar
- 1 tsp vanilla extract
- Cocoa powder or grated vegan chocolate, for dusting

## Other
- None


In [74]:
question_ask_system_prompt = SystemMessagePromptTemplate.from_template("""
You are an experienced teacher who writes quizzes.
Write exactly {count} {question_type} questions at {difficulty} difficulty.
Rules:
- Number the questions 1 to {count}
- For multiple choice, give options A, B, C and D
- Exactly one option is correct
- Do not include answers or explanations
- Do not add any introduction or closing text
""")

question_ask_user_prompt = HumanMessagePromptTemplate.from_template("Write the quiz on this topic: {topic}")

messages = [question_ask_system_prompt,question_ask_user_prompt]

question_template = ChatPromptTemplate(messages)

question_ask_chain = question_template | llm | parser

# questions = question_ask_chain.invoke({"count":5,"question_type":"multiple choice","difficulty":"easy","topic" : "Python lists"})

# print(questions)

answer_prompt = ChatPromptTemplate.from_template("""
Answer each of these quiz questions correctly.
For each one, output:
must be question also
Answer: <correct answer>
Explanation: <one sentence>
Keep the same numbering.

Questions:
{questions}
""")

review_anawer = ChatPromptTemplate.from_template("""
Review these quiz questions and answers.
Say whether each answer is correct. Reply "OK" or "WRONG" for each number,
and also provide me question and answer and Review
with one short reason.provide percentage end 

Questions:
{questions}

Answers:
{answers}
""")

answer_combained_chain = (
    {"questions":question_ask_chain}|
    answer_prompt|
    llm |
    parser
)

answer_review_chain  = (
    {"questions":question_ask_chain,"answers":question_ask_chain}|
    review_anawer |
    llm_groq |
    parser
)


review = answer_review_chain.invoke({"count":5,"question_type":"multiple choice","difficulty":"easy","topic" : "Python lists"})
print(review)



**1. OK**  
- **Question:** What is a Python list?  
- **Answer given:** **A. An ordered collection of items**  
- **Review:** Accurately describes a Python list (ordered, can hold multiple items).  

**2. OK**  
- **Question:** Which symbol is used to create a list in Python?  
- **Answer given:** **C. []**  
- **Review:** Square brackets are the correct literal syntax for a list.  

**3. OK**  
- **Question:** Which of these is a valid Python list?  
- **Answer given:** **B. [1, 2, 3]**  
- **Review:** Brackets with comma‑separated values form a proper list.  

**4. WRONG**  
- **Question:** How do you add an item to the end of a list in Python?  
- **Answer given:** *The answer key shows a different question (“How do you access the first item…”) and options (my_list[0], etc.).*  
- **Review:** The provided answer does not address the “add to end” operation; the correct answer should be **B. append()**.  

**5. WRONG**  
- **Question:** What is the index of the first item in a Python